# Fishrod — Kaggle kurulum defteri

Bu defter sırasıyla şunları yapar:

1. Donanımı yazdırır (çekirdek sayısı, AVX2/BMI2 desteği, RAM, disk, GPU).
2. Rust'ı kurar.
3. Fishrod deposunu GitHub'dan klonlar ve derler.
4. Motor hazırsa `perft` ve `bench` çalıştırır.
5. Ortam bilgisini `/kaggle/working/ortam.txt` dosyasına yazar.

**Başlamadan önce:**

- Sağdaki ayarlardan **Internet: On** yap (telefon doğrulaması gerekir).
- Bu defter için GPU gerekmiyor; GPU kotası eğitim defterlerine kalsın.
- Aşağıdaki `REPO_URL` değerini kendi deponla değiştir.
- Uzun işler için: **Save Version → Save & Run All (Commit)**. Arka planda çalışır, 12 saat içinde bitmelidir.

In [ ]:
REPO_URL = "https://github.com/KULLANICI_ADIN/fishrod.git"  # kendi deponla değiştir
BRANCH = "main"
WORK_DIR = "/tmp/fishrod"         # derleme alanı (oturum bitince silinir)
OUT_DIR = "/kaggle/working"       # kalıcı çıktı alanı (~20 GB, ~500 dosya sınırı)

In [ ]:
import os
import shutil
import subprocess


def sh(cmd):
    # Run a bash command, print its output and return the exit code.
    print(f"$ {cmd}")
    result = subprocess.run(["bash", "-c", "set -o pipefail; " + cmd], text=True, capture_output=True)
    print(result.stdout + result.stderr)
    return result.returncode


def sh_out(cmd):
    # Run a bash command and return its trimmed stdout.
    return subprocess.run(["bash", "-c", cmd], text=True, capture_output=True).stdout.strip()

## 1. Donanım

In [ ]:
sh("nproc")
sh("lscpu | grep -E 'Model name|^CPU\\(s\\)'")
sh("grep -o -w -E 'avx2|bmi2|avx512f' /proc/cpuinfo | sort -u")
sh("free -h")
sh("df -h /kaggle/working /tmp")
sh("nvidia-smi || echo 'GPU yok (CPU oturumu)'")

## 2. Rust kurulumu

In [ ]:
cargo_bin = os.path.expanduser("~/.cargo/bin")
if shutil.which("cargo") is None and not os.path.exists(os.path.join(cargo_bin, "cargo")):
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = cargo_bin + ":" + os.environ["PATH"]
sh("rustc --version && cargo --version")

## 3. Depoyu klonla ve derle

In [ ]:
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
rc = sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORK_DIR}")
assert rc == 0, "Klonlama başarısız: REPO_URL değerini ve Internet ayarını kontrol et."
sh(f"cd {WORK_DIR} && git log -1 --oneline")

In [ ]:
if os.path.exists(os.path.join(WORK_DIR, "Cargo.toml")):
    rc = sh(f'cd {WORK_DIR} && RUSTFLAGS="-C target-cpu=native" cargo build --release 2>&1 | tail -n 25')
    assert rc == 0, "Derleme başarısız; çıktıya bak."
else:
    print("Cargo.toml henüz yok. Faz 1 başlamadan önce bu normal; derleme atlandı.")

## 4. Hızlı kontrol

In [ ]:
BIN = os.path.join(WORK_DIR, "target", "release", "fishrod")
if os.path.exists(BIN):
    sh(f"{BIN} perft 5")   # Faz 1 sonrası: başlangıç pozisyonunda 4865609 olmalı
    sh(f"{BIN} bench")     # Faz 2 sonrası; daha önce hata vermesi normal
else:
    print("Motor ikili dosyası yok; derleme henüz yapılmadı.")

## 5. Ortam raporu

In [ ]:
import datetime

cpu = sh_out("lscpu | grep 'Model name' | sed 's/Model name:[[:space:]]*//'")
simd = sh_out("grep -o -w -E 'avx2|bmi2|avx512f' /proc/cpuinfo | sort -u | tr '\n' ' '")
gpu = sh_out("nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null") or "yok"
lines = [
    f"tarih: {datetime.datetime.now(datetime.timezone.utc).isoformat()}",
    f"nproc: {sh_out('nproc')}",
    f"cpu: {cpu}",
    f"simd: {simd}",
    f"gpu: {gpu}",
    f"rustc: {sh_out('rustc --version')}",
    f"commit: {sh_out(f'cd {WORK_DIR} && git log -1 --oneline')}",
]
os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "ortam.txt"), "w") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))

## Sonraki adımlar

- **Faz 3:** Bu defter kopyalanıp `fishrod_sprt.ipynb` yapılacak: iki commit derlenir, fastchess ile SPRT çalıştırılır, sonuç `/kaggle/working`'e yazılır.
- **Faz 5:** `fishrod_datagen.ipynb` (veri üretimi, CPU) ve `fishrod_egitim.ipynb` (bullet ile eğitim, GPU, kayıt noktasından devam).
- Bir defter sürümünün çıktısını başka bir deftere girdi olarak eklemek için: **Add Data → Notebook Output Files**.